# Model Inference - Kaggle Submission (Best Model: LightGBM)

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

import joblib
import pandas as pd

from src.config import WANDB_ENTITY, WANDB_PROJECT, SUBMISSIONS_DIR
from src.data_loading import load_raw_data, make_submission_frame
from src.wandb_utils import safe_wandb_init, download_model_artifact, log_dataframe_as_artifact
from src.models import WalmartSalesForecaster  

SUBMISSIONS_DIR.mkdir(exist_ok=True, parents=True)

run, artifact_path = download_model_artifact("best-model", alias="latest")
run.finish()
model = joblib.load(f"{artifact_path}/lightgbm_pipeline.joblib")
print("loaded best-model:latest ->", type(model).__name__)

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.


wandb: Currently logged in as: abeku20 (abeku20-free-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: Tracking run with wandb version 0.28.0


wandb: Run data is saved locally in C:\Users\aluda bekurishvili\Desktop\home\projects\store_sales_forecasting\wandb\run-20260712_201912-xljl0d3e
wandb: Run `wandb offline` to turn off syncing.


wandb: Syncing run pious-surf-45


...

wandb: uploading config.yaml; uploading wandb-metadata.json; uploading requirements.txt; uploading output.log


wandb: uploading wandb-metadata.json


wandb:  View run pious-surf-45 at: https://wandb.ai/gchal22-free-university-of-tbilisi-/store_sales_forecast/runs/xljl0d3e
wandb:  View project at: https://wandb.ai/gchal22-free-university-of-tbilisi-/store_sales_forecast
wandb: Synced 5 W&B file(s), 0 media file(s), 0 artifact file(s) and 0 other file(s)


wandb: Find logs at: .\wandb\run-20260712_201912-xljl0d3e\logs


loaded best-model:latest -> WalmartSalesForecaster


## 1. Load Raw Test Data


In [2]:
data = load_raw_data()
test = data["test"]
sample_submission = data["sample_submission"]
print(test.shape)
test.head()

(115064, 4)


,Store,Dept,Date,IsHoliday
0,1,1,2012-11-02,False
1,1,1,2012-11-09,False
2,1,1,2012-11-16,False
3,1,1,2012-11-23,True
4,1,1,2012-11-30,False


## 2. Predict

In [ ]:
predictions = model.predict(test)
print("predicted rows:", len(predictions))
print(f"min/max/mean: {predictions.min():.2f} / {predictions.max():.2f} / {predictions.mean():.2f}")

predicted rows: 115064
min/max/mean: 0.00 / 223948.49 / 16784.25


## 3. Build & Validate Kaggle Submission

In [4]:
submission = make_submission_frame(test, predictions, sample_submission)

assert len(submission) == len(sample_submission), f"row count mismatch: {len(submission)} vs {len(sample_submission)}"
assert set(submission["Id"]) == set(sample_submission["Id"]), "Id set doesn't match sampleSubmission.csv"
assert submission["Weekly_Sales"].isna().sum() == 0, "found missing predictions"
print("submission matches sampleSubmission.csv structure:", len(submission), "rows")

SUBMISSION_PATH = SUBMISSIONS_DIR / "submission_best_model.csv"
submission.to_csv(SUBMISSION_PATH, index=False)
print("saved to", SUBMISSION_PATH)
submission.head()

submission matches sampleSubmission.csv structure: 115064 rows


saved to C:\Users\aluda bekurishvili\Desktop\home\projects\store_sales_forecasting\submissions\submission_best_model.csv


,Id,Weekly_Sales
0,1_1_2012-11-02,35789.862549
1,1_1_2012-11-09,24525.000149
2,1_1_2012-11-16,21901.506560
3,1_1_2012-11-23,24862.353540
4,1_1_2012-11-30,27561.253627


## 4. Log Inference Run to W&B

In [5]:
import wandb

with safe_wandb_init(
    run_name="Best_Model_Inference",
    group="Inference",
    job_type="inference",
) as run:
    log_dataframe_as_artifact(run, submission, "best-model-submission", "submission", SUBMISSION_PATH)
    wandb.log({
        "rows_predicted": len(submission),
        "prediction_min": float(submission["Weekly_Sales"].min()),
        "prediction_max": float(submission["Weekly_Sales"].max()),
        "prediction_mean": float(submission["Weekly_Sales"].mean()),
    })
    print("logged inference run")

wandb: setting up run bdsejfda


wandb: Tracking run with wandb version 0.28.0


wandb: Run data is saved locally in C:\Users\aluda bekurishvili\Desktop\home\projects\store_sales_forecasting\wandb\run-20260712_202118-bdsejfda
wandb: Run `wandb offline` to turn off syncing.


wandb: Syncing run Best_Model_Inference


wandb:  View project at https://wandb.ai/gchal22-free-university-of-tbilisi-/store_sales_forecast


...

wandb: uploading requirements.txt; uploading wandb-summary.json; uploading config.yaml; uploading wandb-metadata.json


wandb: uploading requirements.txt; uploading wandb-summary.json; uploading config.yaml


wandb: 
wandb: Run history:
wandb:  prediction_max ▁
wandb: prediction_mean ▁
wandb:  prediction_min ▁
wandb:  rows_predicted ▁
wandb: 
wandb: Run summary:
wandb:  prediction_max 223948.49012
wandb: prediction_mean 16784.24574
wandb:  prediction_min 0
wandb:  rows_predicted 115064
wandb: 


wandb:  View run Best_Model_Inference at: https://wandb.ai/gchal22-free-university-of-tbilisi-/store_sales_forecast/runs/bdsejfda
wandb:  View project at: https://wandb.ai/gchal22-free-university-of-tbilisi-/store_sales_forecast
wandb: Synced 4 W&B file(s), 0 media file(s), 0 artifact file(s) and 0 other file(s)


wandb: Find logs at: .\wandb\run-20260712_202118-bdsejfda\logs
